In [ ]:
import re
import random
import time
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ── Reproducibility ───────────────────────────────────────────

SEED = 10

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
# Use gpu instead of cpu if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# load Ag News dataset
dataset = load_dataset("sh0416/ag_news")

df_train_full = pd.DataFrame(dataset["train"])
df_test       = pd.DataFrame(dataset["test"])

#Create train test and dev datasets
df_train, df_dev = train_test_split(
    df_train_full, test_size=0.1, random_state=SEED,
    stratify=df_train_full["label"],
)
df_train = df_train.reset_index(drop=True)
df_dev   = df_dev.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Subsample for speed
df_train = df_train.sample(n=5000, random_state=SEED).reset_index(drop=True)
df_dev   = df_dev.sample(n=1000,  random_state=SEED).reset_index(drop=True)
df_test  = df_test.sample(n=1000, random_state=SEED).reset_index(drop=True)

# Put title and description together
for df in [df_train, df_dev, df_test]:
    df["text"] = df["title"] + " " + df["description"]

# Dataset labels
AG_LABELS   = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
NUM_CLASSES = 4
